In [0]:
from pyspark.sql.functions import col, current_timestamp
import requests
import pandas as pd
import io
import re

spark.sql("USE CATALOG scottish_housing_ws")
spark.sql("USE SCHEMA bronze")

In [0]:
url = "https://statistics.gov.scot/downloads/cube-table?uri=http://statistics.gov.scot/data/recorded-crime"

response = requests.get(url)
response.raise_for_status()

df_raw = pd.read_csv(io.StringIO(response.text))

print(df_raw.shape)
print(df_raw.dtypes)
df_raw.head(10)

In [0]:
def clean_col_name(name):
    name = name.lower()
    name = re.sub(r"[^\w]", "_", name)
    name = re.sub(r"_+", "_", name)
    name = name.strip("_")
    return name

df_raw.columns = [clean_col_name(c) for c in df_raw.columns]
print(df_raw.columns.tolist())

In [0]:
df_bronze = spark.createDataFrame(df_raw)
df_bronze.printSchema()
df_bronze.show(10, truncate=False)

In [0]:
(
    df_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("scottish_housing_ws.bronze.crime")
)

In [0]:
# Source: statistics.gov.scot - Recorded Crimes and Offences
# URL: https://statistics.gov.scot/downloads/cube-table?uri=http://statistics.gov.scot/data/recorded-crime
# Covers 1996/97 to 2023/24 by local authority and crime type.
# Includes both raw counts and rates per 10,000 population.
# Open Government Licence.